In [2]:
import pandas as pd
import numpy as np
import os
from dataset_utils import split_data, TimeSeriesDataset, min_max_normalization
from model.lstm import LSTMMdel
from train_utils import Trainer
from evaluate_utils import Evaluator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json
from json_utils import NpEncoder
from typing import List, Tuple
from sklearn.preprocessing import MinMaxScaler

In [2]:
DATASET_PATH = "../../dataset"
INDEX_FIELD = "timestamp"
DATA_FIELD = "num_request"
RESULT_ROOT_PATH="results"
MODEL_NAME="lstm"

N_LOOKBACK = 4
N_PREDICT = 2
N_EPOCHS = 20
BATCH_SIZE=16
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SCHEDULER_MILESTONE = [13, 17]
SCHEDULER_GAMMA = 0.3

In [3]:
def get_data_file_list(dataset_path: str) -> List[str]:
    return os.listdir(dataset_path)


In [4]:
def read_dataset(csv_path: str,index_field:str,data_field:str) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path)
    return df[index_field].to_numpy(), df[data_field].to_numpy()

In [5]:
def learn_on_csv(csv_path: str) -> Tuple[Evaluator, MinMaxScaler, Evaluator, MinMaxScaler, Evaluator, MinMaxScaler]:
    np_index, np_data = read_dataset(csv_path, INDEX_FIELD, DATA_FIELD)
    np_data = np_data.reshape((-1, 1))
    train_set, val_set, test_set = split_data(np_data, 0.6, 0.2, 0.2)
    train_set, train_scaler = min_max_normalization(train_set)
    val_set, val_scaler = min_max_normalization(val_set)
    test_set, test_scaler = min_max_normalization(test_set)
    train_dataset = TimeSeriesDataset(train_set, N_LOOKBACK, N_PREDICT)
    val_dataset = TimeSeriesDataset(val_set, N_LOOKBACK, N_PREDICT)
    test_dataset = TimeSeriesDataset(test_set, N_LOOKBACK, N_PREDICT)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    model = LSTMMdel(1, 64, N_PREDICT, 2).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    # lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, SCHEDULER_MILESTONE, SCHEDULER_GAMMA)
    lr_scheduler = None
    loss_fn = nn.MSELoss()
    trainer = Trainer(model, train_dataloader, loss_fn, optimizer, N_EPOCHS, lr_scheduler, DEVICE)
    trainer.train()
    train_evaluator = Evaluator(model, train_dataloader, loss_fn, DEVICE)
    train_evaluator.evaluate()
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    val_evaluator = Evaluator(model, val_dataloader, loss_fn, DEVICE)
    test_evaluator = Evaluator(model, test_dataloader, loss_fn, DEVICE)
    val_evaluator.evaluate()
    test_evaluator.evaluate()
    return train_evaluator, train_scaler, val_evaluator, val_scaler, test_evaluator, test_scaler

In [6]:
def save_results(x: np.ndarray, file_name: str):
    save_dir = os.path.join(RESULT_ROOT_PATH, MODEL_NAME)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    with open(os.path.join(save_dir, file_name), "w") as f:
        json.dump(x, f, indent=4, cls=NpEncoder)

In [7]:
def save_scaled_results(train_evaluator: Evaluator, val_evaluator: Evaluator,  test_evaluator: Evaluator, file_name: str):
    train_gt_scaled, train_pd_scaled, val_gt_scaled, val_pd_scaled, test_gt_scaled, test_pd_scaled = train_evaluator.get_gt(), train_evaluator.get_pd(), val_evaluator.get_gt(), val_evaluator.get_pd(), test_evaluator.get_gt(), test_evaluator.get_pd()

    save_results(train_gt_scaled, file_name.split(".")[0]+"_gt_train_scaled.json")
    save_results(train_pd_scaled, file_name.split(".")[0]+"_pd_train_scaled.json")
    save_results(val_gt_scaled, file_name.split(".")[0]+"_gt_val_scaled.json")
    save_results(val_pd_scaled, file_name.split(".")[0]+"_pd_val_scaled.json")
    save_results(test_gt_scaled, file_name.split(".")[0]+"_gt_test_scaled.json")
    save_results(test_pd_scaled, file_name.split(".")[0]+"_pd_test_scaled.json")

In [8]:
def save_original_results(train_evaluator: Evaluator, train_scaler: MinMaxScaler, val_evaluator: Evaluator, val_scaler: MinMaxScaler, test_evaluator: Evaluator, test_scaler: MinMaxScaler, file_name: str):
    train_gt_scaled, train_pd_scaled, val_gt_scaled, val_pd_scaled, test_gt_scaled, test_pd_scaled = train_evaluator.get_gt(), train_evaluator.get_pd(), val_evaluator.get_gt(), val_evaluator.get_pd(), test_evaluator.get_gt(), test_evaluator.get_pd()
    
    train_gt_original=np.hstack([train_scaler.inverse_transform(train_gt_scaled[:,i_dim]) for i_dim in range(train_gt_scaled.shape[1])])
    train_gt_original=np.expand_dims(train_gt_original,-1)
    train_pd_original=np.hstack([train_scaler.inverse_transform(train_pd_scaled[:,i_dim]) for i_dim in range(train_pd_scaled.shape[1])])
    train_pd_original=np.expand_dims(train_pd_original,-1)
    val_gt_original=np.hstack([val_scaler.inverse_transform(val_gt_scaled[:,i_dim]) for i_dim in range(val_gt_scaled.shape[1])])
    val_gt_original=np.expand_dims(val_gt_original,-1)
    val_pd_original=np.hstack([val_scaler.inverse_transform(val_pd_scaled[:,i_dim]) for i_dim in range(val_pd_scaled.shape[1])])
    val_pd_original=np.expand_dims(val_pd_original,-1)
    test_gt_original=np.hstack([test_scaler.inverse_transform(test_gt_scaled[:,i_dim]) for i_dim in range(test_gt_scaled.shape[1])])
    test_gt_original=np.expand_dims(test_gt_original,-1)
    test_pd_original=np.hstack([test_scaler.inverse_transform(test_pd_scaled[:,i_dim]) for i_dim in range(test_pd_scaled.shape[1])])
    test_pd_original=np.expand_dims(test_pd_original,-1)

    save_results(train_gt_original, file_name.split(".")[0]+"_gt_train_original.json")
    save_results(train_pd_original, file_name.split(".")[0]+"_pd_train_original.json")
    save_results(val_gt_original, file_name.split(".")[0]+"_gt_val_original.json")
    save_results(val_pd_original, file_name.split(".")[0]+"_pd_val_original.json")
    save_results(test_gt_original, file_name.split(".")[0]+"_gt_test_original.json")
    save_results(test_pd_original, file_name.split(".")[0]+"_pd_test_original.json")

In [9]:
data_file_list = get_data_file_list(DATASET_PATH)
for file_name in data_file_list:
    print("learning on %s" % (file_name))
    train_evaluator, train_scaler, val_evaluator, val_scaler, test_evaluator, test_scaler = learn_on_csv(os.path.join(DATASET_PATH, file_name))
    save_scaled_results(train_evaluator, val_evaluator, test_evaluator, file_name)
    save_original_results(train_evaluator, train_scaler, val_evaluator, val_scaler, test_evaluator, test_scaler, file_name)

learning on workload_1998-06-10.csv
epoch: 1, loss: 0.207001
epoch: 2, loss: 0.103443
epoch: 3, loss: 0.120802
epoch: 4, loss: 0.080201
epoch: 5, loss: 0.062018
epoch: 6, loss: 0.025474
epoch: 7, loss: 0.007562
epoch: 8, loss: 0.005264
epoch: 9, loss: 0.007601
epoch: 10, loss: 0.005632
epoch: 11, loss: 0.004896
epoch: 12, loss: 0.005814
epoch: 13, loss: 0.005578
epoch: 14, loss: 0.004980
epoch: 15, loss: 0.005446
epoch: 16, loss: 0.005341
epoch: 17, loss: 0.005177
epoch: 18, loss: 0.005197
epoch: 19, loss: 0.005461
epoch: 20, loss: 0.005065
learning on workload_1998-06-11.csv
epoch: 1, loss: 0.195204
epoch: 2, loss: 0.101465
epoch: 3, loss: 0.061667
epoch: 4, loss: 0.059673
epoch: 5, loss: 0.048929
epoch: 6, loss: 0.036924
epoch: 7, loss: 0.022698
epoch: 8, loss: 0.010552
epoch: 9, loss: 0.007734
epoch: 10, loss: 0.008289
epoch: 11, loss: 0.007582
epoch: 12, loss: 0.007134
epoch: 13, loss: 0.006816
epoch: 14, loss: 0.006564
epoch: 15, loss: 0.006369
epoch: 16, loss: 0.006184
epoch: 17,